# Keypoint Correspondence Visualization
Visualizes keypoint predictions from Step 1 (zero-shot) and Step 3 (fine-tuned + soft-argmax)
on randomly selected SPair-71k test pairs.

In [ ]:
import os, sys, subprocess

REPO_PATH = '/content/repo'
if not os.path.exists(REPO_PATH):
    subprocess.run([
        'git', 'clone', '--depth', '1',
        '-b', 'local-notebooks',
        'https://github.com/Hesam-AHT/Semantic-Correspondence-with-Visual-Foundation-Models.git',
        REPO_PATH
    ], check=True)
    print('Repo cloned.')
else:
    print('Repo already present.')

os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
print('Ready.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Config — edit these to match your setup ──────────────────────────────────────────
DRIVE_ROOT    = '/content/drive/MyDrive/semantic_correspondence'
SPAIR_BASE    = f'{DRIVE_ROOT}/datasets/SPair-71k/SPair-71k'
RESULTS_DIR   = f'{DRIVE_ROOT}/results'

# Step 1 result file (per_keypoint JSON)
STEP1_KP_FILE = f'{RESULTS_DIR}/step1/exp1_1_dinov2_argmax_per_keypoint.json'

# Step 3 result file (per_keypoint JSON) — set to None if not available yet
STEP3_KP_FILE = f'{RESULTS_DIR}/step3/exp3_softargmax_per_keypoint.json'

# Visualization settings
N_EXAMPLES    = 4        # number of image pairs to show
RANDOM_SEED   = 42       # change for different random selection
CATEGORY      = None     # e.g. 'person', 'dog', 'car' — None for random mix
PCK_THRESHOLD = '0.1'    # '0.05', '0.1', or '0.2'
OUTPUT_PATH   = f'{DRIVE_ROOT}/results/keypoint_grid.png'
# ───────────────────────────────────────────────────────────────────────────

# Auto-detect Step 3
import os
HAS_STEP3 = STEP3_KP_FILE is not None and os.path.exists(STEP3_KP_FILE)
N_COLS = 4 if HAS_STEP3 else 3
print(f'Step 1 file exists: {os.path.exists(STEP1_KP_FILE)}')
print(f'Step 3 file exists: {HAS_STEP3}')
print(f'Columns: {N_COLS}  ({"4-col: Source | GT | Zero-shot | Fine-tuned" if HAS_STEP3 else "3-col: Source | GT | Zero-shot"})')

In [ ]:
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from PIL import Image
from collections import defaultdict
import glob

In [ ]:
def load_per_keypoint(path):
    """Load per_keypoint list from JSON file."""
    with open(path) as f:
        data = json.load(f)
    return data['per_keypoint']


def build_pair_lookup(per_keypoint_list):
    """
    Group keypoints by (category, src_imname, trg_imname).
    Returns: dict {(cat, src, trg): [kp, kp, ...]}
    """
    lookup = defaultdict(list)
    for kp in per_keypoint_list:
        key = (kp['category'], kp['src_imname'], kp['trg_imname'])
        lookup[key].append(kp)
    return lookup


def get_unique_pairs(per_keypoint_list, category=None):
    """Return list of unique (category, src_imname, trg_imname) tuples."""
    seen = set()
    pairs = []
    for kp in per_keypoint_list:
        key = (kp['category'], kp['src_imname'], kp['trg_imname'])
        if key not in seen:
            if category is None or kp['category'] == category:
                seen.add(key)
                pairs.append(key)
    return pairs


def find_annotation_file(spair_base, category, src_name, trg_name):
    """
    Find the SPair-71k annotation JSON for this pair.
    Annotation files are in PairAnnotation/test/ and named like:
    {idx}-{src_stem}-{trg_stem}:{category}.json
    Search by src and trg image names.
    """
    ann_dir = os.path.join(spair_base, 'PairAnnotation', 'test')
    src_stem = os.path.splitext(src_name)[0]
    trg_stem = os.path.splitext(trg_name)[0]
    pattern = os.path.join(ann_dir, f'*{src_stem}*{trg_stem}*.json')
    matches = glob.glob(pattern)
    if matches:
        return matches[0]
    # Try reversed order
    pattern2 = os.path.join(ann_dir, f'*{trg_stem}*{src_stem}*.json')
    matches2 = glob.glob(pattern2)
    return matches2[0] if matches2 else None


def get_src_keypoint(spair_base, category, src_name, trg_name, kp_id):
    """
    Get source keypoint coordinates for a given kp_id from annotation JSON.
    Returns (x, y) or None if not found.
    """
    ann_path = find_annotation_file(spair_base, category, src_name, trg_name)
    if ann_path is None:
        return None
    with open(ann_path) as f:
        ann = json.load(f)
    src_kps = ann.get('src_kps', [])
    kps_ids = ann.get('kps_ids', list(range(len(src_kps))))
    try:
        idx = kps_ids.index(int(kp_id))
        return tuple(src_kps[idx])
    except (ValueError, IndexError):
        try:
            return tuple(src_kps[int(kp_id)])
        except (IndexError, TypeError):
            return None


def draw_marker(ax, x, y, style, img_w, img_h):
    """
    Draw a keypoint marker.
    style: 'gt' | 'correct' | 'wrong_zs' | 'wrong_ft'
    """
    ms = max(9, min(16, img_w // 25))
    if style in ('gt', 'correct'):
        ax.plot(x, y, 'o',
                color='#00EE00',
                markersize=ms,
                markeredgecolor='white',
                markeredgewidth=1.8,
                zorder=10)
    elif style == 'wrong_zs':
        ax.plot(x, y, 'x',
                color='red',
                markersize=ms + 3,
                markeredgewidth=2.8,
                zorder=10)
    elif style == 'wrong_ft':
        ax.plot(x, y, 'x',
                color='#1188FF',
                markersize=ms + 3,
                markeredgewidth=2.8,
                zorder=10)


def pixel_error(pred, gt):
    """Euclidean pixel distance."""
    return float(np.sqrt((pred[0]-gt[0])**2 + (pred[1]-gt[1])**2))

In [ ]:
print('Loading Step 1 keypoints...')
step1_kps  = load_per_keypoint(STEP1_KP_FILE)
step1_lkup = build_pair_lookup(step1_kps)
all_pairs  = get_unique_pairs(step1_kps, category=CATEGORY)
print(f'  {len(step1_kps):,} keypoints across {len(all_pairs):,} pairs')

if HAS_STEP3:
    print('Loading Step 3 keypoints...')
    step3_kps  = load_per_keypoint(STEP3_KP_FILE)
    step3_lkup = build_pair_lookup(step3_kps)
    print(f'  {len(step3_kps):,} keypoints')
else:
    step3_lkup = None
    print('Step 3 not available — showing 3-column layout')

# Select random pairs
random.seed(RANDOM_SEED)
selected = random.sample(all_pairs, min(N_EXAMPLES, len(all_pairs)))
print(f'\nSelected {len(selected)} pairs:')
for cat, src, trg in selected:
    print(f'  [{cat}] {src} → {trg}')

In [ ]:
col_titles = ['Source', 'Ground Truth', 'Zero-shot (DINOv2)']
if HAS_STEP3:
    col_titles.append('Fine-tuned + Soft-argmax')

cell_w, cell_h = 3.2, 3.2
fig_w = cell_w * N_COLS
fig_h = cell_h * N_EXAMPLES + 1.0

fig, axes = plt.subplots(
    N_EXAMPLES, N_COLS,
    figsize=(fig_w, fig_h),
    gridspec_kw={'hspace': 0.05, 'wspace': 0.03}
)
if N_EXAMPLES == 1:
    axes = axes[np.newaxis, :]

# Column headers
for col_idx, title in enumerate(col_titles):
    axes[0, col_idx].set_title(
        title, fontsize=11, fontweight='bold', pad=8
    )

for row_idx, (cat, src_name, trg_name) in enumerate(selected):

    # ── Load images ────────────────────────────────────────────────────────────────
    src_img_path = os.path.join(SPAIR_BASE, 'JPEGImages', cat, src_name)
    trg_img_path = os.path.join(SPAIR_BASE, 'JPEGImages', cat, trg_name)
    src_img = np.array(Image.open(src_img_path).convert('RGB'))
    trg_img = np.array(Image.open(trg_img_path).convert('RGB'))
    ih, iw  = trg_img.shape[:2]

    # ── Get keypoints ────────────────────────────────────────────────────────────
    key = (cat, src_name, trg_name)
    kps1 = step1_lkup.get(key, [])
    kps3 = step3_lkup.get(key, []) if HAS_STEP3 else []
    # Use first keypoint of the pair
    kp1 = kps1[0] if kps1 else None
    kp3 = kps3[0] if kps3 else None

    # Source keypoint from annotation
    src_kp_xy = None
    if kp1 is not None:
        src_kp_xy = get_src_keypoint(
            SPAIR_BASE, cat, src_name, trg_name, kp1['kp_id']
        )

    # ── Column 0: Source ────────────────────────────────────────────────────
    ax = axes[row_idx, 0]
    ax.imshow(src_img)
    if src_kp_xy is not None:
        draw_marker(ax, src_kp_xy[0], src_kp_xy[1],
                    style='gt', img_w=src_img.shape[1], img_h=src_img.shape[0])
    ax.text(0.02, 0.97, cat, transform=ax.transAxes,
            fontsize=8, color='white', va='top',
            bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.55))
    ax.axis('off')

    # ── Column 1: Ground truth ──────────────────────────────────────────────
    ax = axes[row_idx, 1]
    ax.imshow(trg_img)
    if kp1 is not None:
        gt_x, gt_y = kp1['gt']
        draw_marker(ax, gt_x, gt_y, style='gt', img_w=iw, img_h=ih)
    ax.axis('off')

    # ── Column 2: Zero-shot prediction ───────────────────────────────────────
    ax = axes[row_idx, 2]
    ax.imshow(trg_img)
    if kp1 is not None:
        pred_x, pred_y = kp1['pred']
        gt_x,   gt_y   = kp1['gt']
        correct1 = kp1['correct'].get(PCK_THRESHOLD, False)
        err1     = pixel_error(kp1['pred'], kp1['gt'])
        style1   = 'correct' if correct1 else 'wrong_zs'
        draw_marker(ax, pred_x, pred_y, style=style1, img_w=iw, img_h=ih)
        err_color = '#006600' if correct1 else 'red'
        ax.set_xlabel(f'Err: {err1:.1f}px', fontsize=8.5,
                      color=err_color, labelpad=3)
    ax.xaxis.set_label_position('bottom')
    ax.axis('off')
    # Re-enable xlabel after axis('off')
    ax.set_xlabel(ax.get_xlabel(), fontsize=8.5,
                  color=err_color if kp1 else 'black', labelpad=3)
    ax.xaxis.label.set_visible(True)

    # ── Column 3: Fine-tuned + soft-argmax ───────────────────────────────────
    if HAS_STEP3:
        ax = axes[row_idx, 3]
        ax.imshow(trg_img)
        if kp3 is not None:
            pred_x, pred_y = kp3['pred']
            gt_x,   gt_y   = kp3['gt']
            correct3 = kp3['correct'].get(PCK_THRESHOLD, False)
            err3     = pixel_error(kp3['pred'], kp3['gt'])
            style3   = 'correct' if correct3 else 'wrong_ft'
            draw_marker(ax, pred_x, pred_y, style=style3, img_w=iw, img_h=ih)
            err_color3 = '#006600' if correct3 else '#1188FF'
            ax.set_xlabel(f'Err: {err3:.1f}px', fontsize=8.5,
                          color=err_color3, labelpad=3)
        ax.xaxis.set_label_position('bottom')
        ax.axis('off')
        ax.set_xlabel(ax.get_xlabel(), fontsize=8.5,
                      color=err_color3 if kp3 else 'black', labelpad=3)
        ax.xaxis.label.set_visible(True)

# ── Overall title ───────────────────────────────────────────────────────────────
cat_str = f'({CATEGORY.capitalize()} Category)' if CATEGORY else '(Mixed Categories)'
fig.suptitle(
    f'Keypoint Localization Examples: Easy vs Hard Keypoints\n{cat_str}',
    fontsize=13, fontweight='bold', y=1.005
)

plt.savefig(OUTPUT_PATH, dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print(f'\nSaved to: {OUTPUT_PATH}')

## Notes
- **Green dot**: correct prediction (within PCK threshold)
- **Red ✗**: wrong prediction (zero-shot)
- **Blue ✗**: wrong prediction (fine-tuned)
- Error shown in pixels below each prediction column
- Change `RANDOM_SEED` in the config cell to see different pairs
- Change `CATEGORY` to filter by object category (e.g. `'person'`, `'dog'`)